# Beyond-ECD Comparison

## Protocol

**Status:** fixed on 2026-10-16, before any model in this notebook was run. Analysis A of `ANALYSIS_PLAN.md` (Amendment 1), recorded in git before execution.

### 1. Framing

The research aim is to test whether multimodal information (slit-lamp findings, Konan morphology, specular microscopy images) improves donor cornea grading beyond ECD alone. This notebook makes that comparison on the primary cohort (non-2023 tissues).

Context that informed this analysis:

- The primary cohort was adopted after the results of notebooks 08–09 were known (see `ANALYSIS_PLAN.md`, timing disclosure).
- A descriptive check in `CLAIMS_AUDIT.md` found that a two-threshold ECD rule scored macro F1 0.800 on these tissues, against 0.758 for the tabular model. This analysis replaces that unmatched check with a matched, donor-grouped comparison against a learned ECD-only model.

### 2. Research question

Do the tabular and hybrid models improve on ECD alone when all are evaluated on the same unseen donors?

### 3. Cohort

| Set | Definition | Expected |
|---|---|---|
| Primary cohort | Labelled, image-matched (two review tissues excluded), year in `Tissue_ID` ≠ 2023; as in notebook 09 | 551 tissues, 312 donors |
| **Evaluation set** | Primary-cohort tissues with a non-missing `ECD_CD_cells_per_mm2` | **470 tissues, 280 donors**; A+ 244, A 159, B 67; years 2020: 78, 2021: 11, 2022: 54, 2024: 155, 2025: 172 |
| No-ECD set | Primary-cohort tissues without ECD (all Konan features missing) | 81 tissues; A+ 33, A 37, B 11 |

The notebook asserts these counts and stops if any differ.

### 4. Models

All models use the Random Forest hyperparameters from notebooks 07–09 (`n_estimators=300, max_depth=15, min_samples_split=5, min_samples_leaf=1, max_features="sqrt", class_weight="balanced", random_state=42`). No tuning.

| Model | Inputs | Training tissues in each fold |
|---|---|---|
| **ECD-only** | `ECD_CD_cells_per_mm2` alone, no preprocessing | Training-fold tissues with ECD |
| Tabular | 24 features via `ms.make_preprocessor()` (as notebook 09) | All training-fold tissues of the primary cohort |
| Hybrid | Tabular + frozen ResNet-50 embeddings → StandardScaler → PCA 16 (as notebook 09) | All training-fold tissues of the primary cohort |

- Using the same learner for every model means the comparison varies only the input information.
- The ECD-only model cannot learn from tissues without ECD, so they are excluded from its training.
- Tabular and Hybrid are refitted with notebook 09's exact settings and folds.

**Reproducibility check:** the refitted Tabular and Hybrid out-of-fold predictions must match notebook 09's saved predictions (`non_2023_oof_predictions.csv`) for every tissue and repeat. If they do not, the notebook stops and the discrepancy is recorded in the deviations log before anything else is done.

### 5. Validation

- Existing folds `Fold_r0` … `Fold_r4` from `donor_grouped_folds.csv`, restricted to the primary cohort. All fitting, including PCA, uses the training fold only.
- Each model predicts every held-out tissue in the primary cohort that it can. Primary metrics are computed on the evaluation-set tissues within each held-out fold.
- Checks before fitting:
  - no donor in both training and test of any fold;
  - every primary-cohort tissue is tested exactly once per repeat;
  - every held-out fold contains at least one evaluation-set tissue of each grade.

### 6. Comparisons and metrics

| Role | Comparison | Metric |
|---|---|---|
| **Primary P1** | Tabular − ECD-only | Macro F1, per fold, on evaluation-set tissues |
| **Primary P2** | Hybrid − ECD-only | Macro F1, per fold, on evaluation-set tissues |
| Secondary | P1 and P2 | Balanced accuracy, accuracy, grade B recall (unadjusted) |
| Descriptive | Hybrid − Tabular on the evaluation set | Macro F1 (notebook 09 is the test of this comparison; no new test here) |
| Descriptive | All models | Per-grade precision and recall, pooled over repeats |
| Descriptive | ECD threshold rule (Section 8) | Same metrics, no tests |
| Descriptive | Tabular and Hybrid on the no-ECD set | Macro F1 and per-grade recall, pooled over repeats |

### 7. Statistical analysis

- For P1 and P2: mean paired fold-level difference, corrected 95% CI and corrected repeated-CV t-test (Nadeau & Bengio, 2003; Bouckaert & Frank, 2004), 24 degrees of freedom.
  - The variance correction uses n_test / n_train from mean primary-cohort fold sizes.
- **Multiplicity:** Holm correction across P1 and P2 p-values. Confidence intervals are not multiplicity-adjusted and are reported as such.
- **Per-year (descriptive):** for P1 and P2, the pooled difference by year and with each year excluded in turn. 2021 (11 tissues) is flagged as too small to interpret.

### 8. ECD threshold rule (descriptive only)

Rule: B if ECD < 2250, A if 2250 ≤ ECD < 2500, A+ if ECD ≥ 2500.

**Threshold source:**

- These thresholds were **inferred from the 2023 records** after those records had been inspected (2023 A+ minimum ECD 2500; 2023 B maximum 2247).
- No documented eye-bank grading criteria were available when this protocol was written.
- The rule is not fitted to the evaluation tissues but was chosen post hoc, so it is reported **descriptively only**: no tests, and no conclusions rest on it.
- If documented eye-bank criteria are obtained later, they will be evaluated under a separate protocol.

### 9. Interpretation rules

Fixed before running and applied in code.

**Each primary comparison (P1, P2):**

| Result | Reported as |
|---|---|
| Difference > 0 and Holm-adjusted p < 0.05 | Model improved on ECD alone |
| Difference < 0 and Holm-adjusted p < 0.05 | Model performed worse than ECD alone |
| Otherwise | Inconclusive: no evidence of a difference from ECD alone |

**Answer to the research question:**

- *Evidence of improvement beyond ECD alone*: at least one of P1 or P2 is reported as improved.
- *No evidence of improvement beyond ECD alone in this cohort*: neither is improved and neither is worse.
- *Models performed worse than ECD alone*: either is worse. The specific model or models are named.

**Qualifiers:**

- **Grade B recall:** for each comparison, if the mean fold-level grade B recall difference is below −0.02, the result is reported with the reduced B recall and that model is not recommended over ECD alone. As in notebook 09, −0.02 is a project-defined decision rule, not a clinically validated threshold.
- **Per-year:** if a positive pooled difference becomes zero or negative when any single year is excluded, the result is reported as dependent on that year.

**Consequence for analysis B (explainability):** unless an improvement is found, explanations describe model behaviour, not evidence that sources other than ECD add information.

**Other rules:**

- **No changes after results:** no other comparator models, learners, hyperparameters, cohorts, rule thresholds or metrics. Any further analysis is labelled post hoc.
- **Errors:** a code error may be fixed and the notebook rerun only if the fix does not change a protocol choice. Every rerun is logged in the deviations log with its reason.

# Analysis

Implements the protocol above without changes. Section numbers refer to the protocol.

In [1]:
import hashlib
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

sys.path.insert(0, str(Path("../../scripts").resolve()))
import modeling_setup as ms

SEED = ms.RANDOM_STATE
ECD = "ECD_CD_cells_per_mm2"
IMAGE_DATA_DIR = ms.PROCESSED_DIR.parent.parent / "image" / "data" / "processed"
EMBEDDING_PATH = IMAGE_DATA_DIR / "embeddings" / "resnet50_imagenet_fullres.npz"
NB09_OOF_PATH = ms.PROCESSED_DIR / "non_2023_oof_predictions.csv"
REVIEW_IDS = ["NEB-2024-08-16259 R", "NEB-2025-09-17471 R"]
PCA_COMPONENTS = 16

# Protocol Section 4.
RF_PARAMS = dict(
    n_estimators=300, max_depth=15, min_samples_split=5, min_samples_leaf=1,
    max_features="sqrt", class_weight="balanced", random_state=SEED, n_jobs=-1,
)

# Protocol Section 3.
EXPECTED_PRIMARY = {"tissues": 551, "donors": 312}
EXPECTED_EVAL = {
    "tissues": 470, "donors": 280,
    "grades": {"A+": 244, "A": 159, "B": 67},
    "years": {2020: 78, 2021: 11, 2022: 54, 2024: 155, 2025: 172},
}
EXPECTED_NO_ECD = {"tissues": 81, "grades": {"A+": 33, "A": 37, "B": 11}}

# Provenance of the input data (not part of the protocol; recorded for audit).
for path in [ms.DATA_PATH, ms.DONOR_FOLDS_PATH, EMBEDDING_PATH, NB09_OOF_PATH]:
    print(f"{path.name}: sha256 {hashlib.sha256(path.read_bytes()).hexdigest()[:16]}…")

cleaned_grading_dataset.xlsx: sha256 0ce9b384a51a4c47…
donor_grouped_folds.csv: sha256 755280ff9d7bd388…
resnet50_imagenet_fullres.npz: sha256 883d4a1723a17be7…
non_2023_oof_predictions.csv: sha256 07580d83a171e636…


## Cohort (Protocol Section 3)

In [2]:
def normalize_id(value):
    return " ".join(str(value).split()).replace("NEB- ", "NEB-").casefold()

labelled = ms.load_labelled_data()
folds = ms.get_donor_folds(labelled)

manifest = pd.read_csv(IMAGE_DATA_DIR / "clean_image_manifest.csv")
manifest["Match_ID"] = manifest["Tissue_ID"].map(normalize_id)
manifest = manifest[~manifest["Match_ID"].isin({normalize_id(i) for i in REVIEW_IDS})]

labelled["Match_ID"] = labelled["Tissue_ID"].map(normalize_id)
labelled["Year"] = labelled["Tissue_ID"].str.extract(r"NEB-\s*(\d{4})")[0].astype("Int64")

with_image = labelled.merge(manifest[["Match_ID"]], on="Match_ID", how="inner", validate="one_to_one")
cohort = (
    with_image[with_image["Year"].notna() & (with_image["Year"] != 2023)]
    .merge(folds, on="Tissue_ID", how="left", validate="one_to_one")
    .reset_index(drop=True)
)
cohort["Year"] = cohort["Year"].astype(int)

has_ecd = cohort[ECD].notna().to_numpy()
evaluation = cohort[has_ecd]
no_ecd = cohort[~has_ecd]


def describe(frame):
    return {
        "tissues": len(frame),
        "donors": frame["Donor_Group"].nunique(),
        "grades": {k: int(v) for k, v in frame[ms.TARGET].value_counts().items()},
        "years": {int(k): int(v) for k, v in frame["Year"].value_counts().sort_index().items()},
    }

primary_desc, eval_desc, no_ecd_desc = describe(cohort), describe(evaluation), describe(no_ecd)
print("Primary cohort:", primary_desc)
print("Evaluation set:", eval_desc)
print("No-ECD set:", no_ecd_desc)

assert {k: primary_desc[k] for k in EXPECTED_PRIMARY} == EXPECTED_PRIMARY, "Primary cohort differs from protocol"
assert {k: eval_desc[k] for k in EXPECTED_EVAL} == EXPECTED_EVAL, "Evaluation set differs from protocol"
assert {k: no_ecd_desc[k] for k in EXPECTED_NO_ECD} == EXPECTED_NO_ECD, "No-ECD set differs from protocol"
assert no_ecd[ms.NUMERIC_FEATURES].isna().all(axis=1).all(), "No-ECD tissues should lack all Konan features"
print("\nAll sets match the protocol.")

Primary cohort: {'tissues': 551, 'donors': 312, 'grades': {'A+': 277, 'A': 196, 'B': 78}, 'years': {2020: 128, 2021: 12, 2022: 58, 2024: 168, 2025: 185}}
Evaluation set: {'tissues': 470, 'donors': 280, 'grades': {'A+': 244, 'A': 159, 'B': 67}, 'years': {2020: 78, 2021: 11, 2022: 54, 2024: 155, 2025: 172}}
No-ECD set: {'tissues': 81, 'donors': 58, 'grades': {'A': 37, 'A+': 33, 'B': 11}, 'years': {2020: 50, 2021: 1, 2022: 4, 2024: 13, 2025: 13}}

All sets match the protocol.


In [3]:
cached = np.load(EMBEDDING_PATH, allow_pickle=True)
row_of = {tissue_id: i for i, tissue_id in enumerate(cached["tissue_ids"])}
assert all(t in row_of for t in cohort["Tissue_ID"]), "Missing cached embeddings"
embeddings = cached["embeddings"][[row_of[t] for t in cohort["Tissue_ID"]]]

EMBEDDING_COLUMNS = [f"emb_{i}" for i in range(embeddings.shape[1])]
X = pd.concat([
    cohort[ms.ALL_FEATURES].reset_index(drop=True),
    pd.DataFrame(embeddings, columns=EMBEDDING_COLUMNS),
], axis=1)
y = cohort[ms.TARGET].to_numpy()
years = cohort["Year"].to_numpy()
print("Feature matrix:", X.shape)

Feature matrix: (551, 2072)


## Models (Protocol Section 4)

In [4]:
def make_arm(inputs):
    if inputs == "ECD-only":
        branches = [("ecd", "passthrough", [ECD])]
    else:
        branches = [("tabular", ms.make_preprocessor(), ms.ALL_FEATURES)]
        if inputs == "Hybrid":
            branches.append(("image", Pipeline([
                ("scaler", StandardScaler()),
                ("pca", PCA(n_components=PCA_COMPONENTS, random_state=SEED)),
            ]), EMBEDDING_COLUMNS))
    return Pipeline([
        ("features", ColumnTransformer(branches, remainder="drop")),
        ("classifier", RandomForestClassifier(**RF_PARAMS)),
    ])

ARMS = {name: make_arm(name) for name in ["ECD-only", "Tabular", "Hybrid"]}


def ecd_rule(values):
    # Protocol Section 8: thresholds inferred from 2023 records; descriptive only.
    return np.where(values >= 2500, "A+", np.where(values >= 2250, "A", "B"))

## Validation checks (Protocol Section 5)

All checks must pass before any model is fitted.

In [5]:
fold_splits = {
    repeat: list(ms.iter_donor_folds(folds, cohort["Tissue_ID"], repeat))
    for repeat in range(ms.N_REPEATS)
}

for repeat, splits in fold_splits.items():
    tested = np.sort(np.concatenate([test for _, test in splits]))
    assert np.array_equal(tested, np.arange(len(cohort))), "Each tissue must be tested once per repeat"
    for train, test in splits:
        assert set(cohort.loc[train, "Donor_Group"]).isdisjoint(cohort.loc[test, "Donor_Group"]), "Donor leak"
        assert set(y[test[has_ecd[test]]]) == set(ms.CLASSES), "A held-out fold lacks a grade in the evaluation set"

sizes = np.array([(len(tr), len(te)) for s in fold_splits.values() for tr, te in s])
N_TRAIN, N_TEST = sizes.mean(axis=0)
print("All validation checks passed.")
print(f"Primary-cohort fold sizes: train {sizes[:, 0].min()}–{sizes[:, 0].max()}, test {sizes[:, 1].min()}–{sizes[:, 1].max()}")

All validation checks passed.
Primary-cohort fold sizes: train 428–450, test 101–123


## Cross-validated predictions

In [6]:
oof = {(arm, r): np.full(len(cohort), None, dtype=object) for arm in ARMS for r in range(ms.N_REPEATS)}

start = time.time()
for repeat, splits in fold_splits.items():
    for train, test in splits:
        for arm, pipeline in ARMS.items():
            if arm == "ECD-only":
                train_rows, test_rows = train[has_ecd[train]], test[has_ecd[test]]
            else:
                train_rows, test_rows = train, test
            model = clone(pipeline).fit(X.iloc[train_rows], y[train_rows])
            oof[(arm, repeat)][test_rows] = model.predict(X.iloc[test_rows])

print(f"{len(sizes) * len(ARMS)} model fits in {(time.time() - start) / 60:.1f} min")

75 model fits in 0.9 min


### Reproducibility check (Protocol Section 4)

Refitted Tabular and Hybrid predictions must match notebook 09 exactly.

In [7]:
saved = pd.read_csv(NB09_OOF_PATH)
for arm in ["Tabular", "Hybrid"]:
    ours = pd.DataFrame({
        "Tissue_ID": np.tile(cohort["Tissue_ID"], ms.N_REPEATS),
        "Repeat": np.repeat(range(ms.N_REPEATS), len(cohort)),
        "Ours": np.concatenate([oof[(arm, r)] for r in range(ms.N_REPEATS)]),
    }).merge(saved[["Tissue_ID", "Repeat", arm]], on=["Tissue_ID", "Repeat"], how="left", validate="one_to_one")
    agreement = (ours["Ours"] == ours[arm]).mean()
    print(f"{arm}: {agreement:.2%} of {len(ours)} predictions match notebook 09")
    assert agreement == 1.0, f"{arm} predictions differ from notebook 09: stop and record in the deviations log"

Tabular: 100.00% of 2755 predictions match notebook 09
Hybrid: 100.00% of 2755 predictions match notebook 09


## Fold-level metrics on the evaluation set (Protocol Section 6)

In [8]:
METRICS = {
    "Macro F1": lambda t, p: f1_score(t, p, labels=ms.CLASSES, average="macro", zero_division=0),
    "Balanced accuracy": balanced_accuracy_score,
    "Accuracy": accuracy_score,
    "Recall B": lambda t, p: recall_score(t, p, labels=["B"], average="macro", zero_division=0),
}

rule_predictions = np.full(len(cohort), None, dtype=object)
rule_predictions[has_ecd] = ecd_rule(cohort.loc[has_ecd, ECD].to_numpy())

fold_rows = []
for repeat, splits in fold_splits.items():
    for fold, (_, test) in enumerate(splits):
        rows = test[has_ecd[test]]
        predictors = {arm: oof[(arm, repeat)][rows] for arm in ARMS}
        predictors["ECD rule"] = rule_predictions[rows]
        for name, pred in predictors.items():
            fold_rows.append({
                "Model": name, "Repeat": repeat, "Fold": fold, "N": len(rows),
                **{m: fn(y[rows], pred.astype(str)) for m, fn in METRICS.items()},
            })

fold_scores = pd.DataFrame(fold_rows)
MODELS = ["ECD-only", "Tabular", "Hybrid", "ECD rule"]
print("Evaluation-set metrics, mean and SD over 25 folds (ECD rule descriptive only):")
print(fold_scores.groupby("Model")[list(METRICS)].agg(["mean", "std"]).reindex(MODELS).round(4).to_string())

Evaluation-set metrics, mean and SD over 25 folds (ECD rule descriptive only):
         Macro F1         Balanced accuracy         Accuracy         Recall B        
             mean     std              mean     std     mean     std     mean     std
Model                                                                                
ECD-only   0.7073  0.0509            0.7114  0.0523   0.7093  0.0491   0.7929  0.1104
Tabular    0.7557  0.0423            0.7533  0.0434   0.7582  0.0403   0.8140  0.1017
Hybrid     0.7886  0.0412            0.7821  0.0445   0.7965  0.0325   0.8089  0.1272
ECD rule   0.7973  0.0373            0.7694  0.0424   0.8064  0.0289   0.7901  0.1171


## Primary comparisons P1 and P2 (Protocol Section 7)

In [9]:
def corrected_ttest(differences):
    d = np.asarray(differences, dtype=float)
    n = len(d)
    se = np.sqrt((1 / n + N_TEST / N_TRAIN) * d.var(ddof=1))
    t = d.mean() / se
    p = 2 * stats.t.sf(abs(t), df=n - 1)
    half = stats.t.ppf(0.975, df=n - 1) * se
    return d.mean(), d.mean() - half, d.mean() + half, t, p


def holm(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted = np.empty_like(p_values)
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (len(p_values) - rank) * p_values[idx])
        adjusted[idx] = min(running, 1.0)
    return adjusted


wide = fold_scores.pivot_table(index=["Repeat", "Fold"], columns="Model", values=list(METRICS))
COMPARISONS = {"P1": ("Tabular", "ECD-only"), "P2": ("Hybrid", "ECD-only")}

rows = []
for label, (model, baseline) in COMPARISONS.items():
    for metric in METRICS:
        d = wide[(metric, model)] - wide[(metric, baseline)]
        mean, low, high, t, p = corrected_ttest(d)
        rows.append({
            "Comparison": f"{label}: {model} − {baseline}",
            "Role": "Primary" if metric == "Macro F1" else "Secondary",
            "Metric": metric,
            "Model": wide[(metric, model)].mean(),
            "ECD-only": wide[(metric, baseline)].mean(),
            "Difference": mean, "CI low": low, "CI high": high, "t": t, "p": p,
            "Folds improved": f"{(d > 0).sum()}/{len(d)}",
        })
results = pd.DataFrame(rows)

primary_mask = results["Role"] == "Primary"
results["p (Holm)"] = np.nan
results.loc[primary_mask, "p (Holm)"] = holm(results.loc[primary_mask, "p"])

print("Primary (macro F1; Holm across P1 and P2; CIs not multiplicity-adjusted):")
print(results[primary_mask].drop(columns="Role").round(4).to_string(index=False))
print("\nSecondary (unadjusted):")
print(results[~primary_mask].drop(columns=["Role", "p (Holm)"]).round(4).to_string(index=False))

Primary (macro F1; Holm across P1 and P2; CIs not multiplicity-adjusted):
            Comparison   Metric  Model  ECD-only  Difference  CI low  CI high      t      p Folds improved  p (Holm)
P1: Tabular − ECD-only Macro F1 0.7557    0.7073      0.0484  0.0050   0.0918 2.3016 0.0303          22/25    0.0303
 P2: Hybrid − ECD-only Macro F1 0.7886    0.7073      0.0813  0.0309   0.1317 3.3265 0.0028          25/25    0.0056

Secondary (unadjusted):
            Comparison            Metric  Model  ECD-only  Difference  CI low  CI high      t      p Folds improved
P1: Tabular − ECD-only Balanced accuracy 0.7533    0.7114      0.0419  0.0068   0.0770 2.4608 0.0214          24/25
P1: Tabular − ECD-only          Accuracy 0.7582    0.7093      0.0490  0.0035   0.0944 2.2241 0.0358          22/25
P1: Tabular − ECD-only          Recall B 0.8140    0.7929      0.0210 -0.0202   0.0622 1.0532 0.3027           7/25
 P2: Hybrid − ECD-only Balanced accuracy 0.7821    0.7114      0.0707  0.0219   0.1195

## Descriptive results (Protocol Section 6)

In [10]:
d = wide[("Macro F1", "Hybrid")] - wide[("Macro F1", "Tabular")]
print(f"Hybrid − Tabular on the evaluation set (descriptive; tested in notebook 09): "
      f"mean fold-level macro F1 difference {d.mean():+.4f}, improved in {(d > 0).sum()}/{len(d)} folds")

d = wide[("Macro F1", "ECD rule")] - wide[("Macro F1", "ECD-only")]
print(f"ECD rule − ECD-only (descriptive; thresholds inferred from 2023 records): "
      f"mean fold-level macro F1 difference {d.mean():+.4f}")

Hybrid − Tabular on the evaluation set (descriptive; tested in notebook 09): mean fold-level macro F1 difference +0.0329, improved in 20/25 folds
ECD rule − ECD-only (descriptive; thresholds inferred from 2023 records): mean fold-level macro F1 difference +0.0900


In [11]:
eval_rows = np.flatnonzero(has_ecd)
y_eval = np.tile(y[eval_rows], ms.N_REPEATS)
pooled = {arm: np.concatenate([oof[(arm, r)][eval_rows] for r in range(ms.N_REPEATS)]).astype(str) for arm in ARMS}
pooled["ECD rule"] = np.tile(rule_predictions[eval_rows], ms.N_REPEATS).astype(str)

print("Per-grade precision and recall on the evaluation set, pooled over repeats:")
print(pd.concat({
    name: pd.DataFrame({
        "Precision": precision_score(y_eval, pred, labels=ms.CLASSES, average=None, zero_division=0),
        "Recall": recall_score(y_eval, pred, labels=ms.CLASSES, average=None, zero_division=0),
    }, index=ms.CLASSES)
    for name, pred in pooled.items()
}, axis=1).round(3).to_string())

Per-grade precision and recall on the evaluation set, pooled over repeats:
    ECD-only          Tabular           Hybrid         ECD rule       
   Precision Recall Precision Recall Precision Recall Precision Recall
A+     0.761  0.785     0.784  0.846     0.809  0.890     0.749  0.992
A      0.594  0.558     0.676  0.599     0.745  0.649     0.894  0.528
B      0.773  0.794     0.830  0.818     0.860  0.806     1.000  0.791


### Per-year results (descriptive)

2021 (11 tissues) is too small to interpret.

In [12]:
macro_f1 = METRICS["Macro F1"]
years_eval = np.tile(years[eval_rows], ms.N_REPEATS)
eval_years = sorted(EXPECTED_EVAL["years"])

by_year, leave_out = [], []
for label, (model, baseline) in COMPARISONS.items():
    for year in eval_years:
        sel = years_eval == year
        by_year.append({
            "Comparison": label, "Year": year, "Tissues": EXPECTED_EVAL["years"][year],
            model: macro_f1(y_eval[sel], pooled[model][sel]),
            "ECD-only": macro_f1(y_eval[sel], pooled[baseline][sel]),
            "Difference": macro_f1(y_eval[sel], pooled[model][sel]) - macro_f1(y_eval[sel], pooled[baseline][sel]),
        })
        keep = years_eval != year
        leave_out.append({
            "Comparison": label, "Excluded year": year,
            "Difference": macro_f1(y_eval[keep], pooled[model][keep]) - macro_f1(y_eval[keep], pooled[baseline][keep]),
        })

by_year = pd.DataFrame(by_year)
leave_out = pd.DataFrame(leave_out)
overall_pooled = {
    label: macro_f1(y_eval, pooled[model]) - macro_f1(y_eval, pooled[baseline])
    for label, (model, baseline) in COMPARISONS.items()
}

for label in COMPARISONS:
    print(f"{label}: overall pooled difference {overall_pooled[label]:+.4f}")
    print(by_year[by_year["Comparison"] == label].drop(columns="Comparison").dropna(axis=1, how="all").round(4).to_string(index=False))
    print(leave_out[leave_out["Comparison"] == label].drop(columns="Comparison").round(4).to_string(index=False), "\n")

P1: overall pooled difference +0.0469
 Year  Tissues  Tabular  ECD-only  Difference
 2020       78   0.7378    0.6907      0.0471
 2021       11   0.7593    0.6459      0.1134
 2022       54   0.7826    0.7634      0.0191
 2024      155   0.7500    0.6695      0.0805
 2025      172   0.7455    0.7134      0.0321
 Excluded year  Difference
          2020      0.0480
          2021      0.0454
          2022      0.0507
          2024      0.0348
          2025      0.0581 

P2: overall pooled difference +0.0803
 Year  Tissues  ECD-only  Difference  Hybrid
 2020       78    0.6907      0.0577  0.7483
 2021       11    0.6459      0.1346  0.7805
 2022       54    0.7634      0.0166  0.7800
 2024      155    0.6695      0.1132  0.7827
 2025      172    0.7134      0.0802  0.7937
 Excluded year  Difference
          2020      0.0847
          2021      0.0781
          2022      0.0886
          2024      0.0669
          2025      0.0817 



### No-ECD set (descriptive)

The ECD-only model and the rule cannot grade these tissues.

In [13]:
no_ecd_rows = np.flatnonzero(~has_ecd)
y_no = np.tile(y[no_ecd_rows], ms.N_REPEATS)
for arm in ["Tabular", "Hybrid"]:
    pred = np.concatenate([oof[(arm, r)][no_ecd_rows] for r in range(ms.N_REPEATS)]).astype(str)
    recalls = recall_score(y_no, pred, labels=ms.CLASSES, average=None, zero_division=0)
    print(f"{arm}: macro F1 {macro_f1(y_no, pred):.4f}; recall "
          + ", ".join(f"{c} {r:.3f}" for c, r in zip(ms.CLASSES, recalls)))

Tabular: macro F1 0.4064; recall A+ 0.485, A 0.557, B 0.164
Hybrid: macro F1 0.4679; recall A+ 0.788, A 0.454, B 0.164


## Interpretation (Protocol Section 9)

The rules are applied mechanically to the results above.

In [14]:
primary = results[primary_mask].set_index("Comparison")
b_recall = results[results["Metric"] == "Recall B"].set_index("Comparison")["Difference"]

verdicts = {}
for comparison, row in primary.iterrows():
    label = comparison.split(":")[0]
    if row["Difference"] > 0 and row["p (Holm)"] < 0.05:
        verdicts[label] = "improved"
    elif row["Difference"] < 0 and row["p (Holm)"] < 0.05:
        verdicts[label] = "worse"
    else:
        verdicts[label] = "inconclusive"

    print(f"{comparison}: macro F1 {row['Difference']:+.4f} [{row['CI low']:+.4f}, {row['CI high']:+.4f}], "
          f"p = {row['p']:.4f}, Holm p = {row['p (Holm)']:.4f} -> {verdicts[label]}")

    b_difference = b_recall[comparison]
    print(f"  Grade B recall qualifier: {'TRIGGERED' if b_difference < -0.02 else 'not triggered'} "
          f"(mean B recall difference {b_difference:+.4f}; threshold −0.02)")

    flagged = leave_out.loc[(leave_out["Comparison"] == label) & (leave_out["Difference"] <= 0), "Excluded year"].tolist()
    year_dependent = overall_pooled[label] > 0 and bool(flagged)
    print(f"  Per-year qualifier: {'TRIGGERED, depends on ' + str(flagged) if year_dependent else 'not triggered'}")

worse = [label for label, v in verdicts.items() if v == "worse"]
if worse:
    answer = f"Models performed worse than ECD alone: {', '.join(COMPARISONS[l][0] for l in worse)}"
elif "improved" in verdicts.values():
    answer = "Evidence of improvement beyond ECD alone"
else:
    answer = "No evidence of improvement beyond ECD alone in this cohort"

print(f"\nAnswer to the research question: {answer}.")
if answer != "Evidence of improvement beyond ECD alone":
    print("Consequence for analysis B: explanations describe model behaviour, not evidence that "
          "sources other than ECD add information.")

P1: Tabular − ECD-only: macro F1 +0.0484 [+0.0050, +0.0918], p = 0.0303, Holm p = 0.0303 -> improved
  Grade B recall qualifier: not triggered (mean B recall difference +0.0210; threshold −0.02)
  Per-year qualifier: not triggered
P2: Hybrid − ECD-only: macro F1 +0.0813 [+0.0309, +0.1317], p = 0.0028, Holm p = 0.0056 -> improved
  Grade B recall qualifier: not triggered (mean B recall difference +0.0160; threshold −0.02)
  Per-year qualifier: not triggered

Answer to the research question: Evidence of improvement beyond ECD alone.


In [15]:
fold_scores.to_csv(ms.PROCESSED_DIR / "beyond_ecd_fold_scores.csv", index=False)
results.to_csv(ms.PROCESSED_DIR / "beyond_ecd_results.csv", index=False)
print("Saved fold scores and comparison results to", ms.PROCESSED_DIR)

Saved fold scores and comparison results to C:\Users\pulit\OneDrive\Desktop\University\Y4S1\research\xai-cornea-grading\ml\grading\data\processed


## Deviations Log

Record any rerun or departure from the protocol here, with date and reason, before reporting results.

| Date | Change | Reason |
|---|---|---|
| — | None | — |